In [6]:
import requests
from decimal import Decimal, getcontext

getcontext().prec = 28

def get_sol_market_data():
    url = "https://api.coingecko.com/api/v3/coins/markets"
    params = {
        "vs_currency": "usd",
        "ids": "solana"
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if not data:
        raise RuntimeError("CoinGecko returned empty response")

    row = data[0]

    price = Decimal(str(row["current_price"]))
    market_cap = Decimal(str(row["market_cap"]))
    circulating_supply = Decimal(str(row["circulating_supply"]))

    return {
        "name": row["name"],
        "symbol": row["symbol"],
        "price_usd": price,
        "market_cap_usd": market_cap,
        "circulating_supply": circulating_supply,
    }

def compute_from_mvrv(price, market_cap, circulating_supply, mvrv):
    mvrv = Decimal(str(mvrv))

    if mvrv <= 0:
        raise ValueError("MVRV must be > 0")

    realized_cap = market_cap / mvrv
    realized_price = price / mvrv
    discount_to_realized = (price / realized_price) - Decimal("1")

    return {
        "mvrv": mvrv,
        "realized_cap_usd": realized_cap,
        "realized_price_usd": realized_price,
        "discount_to_realized": discount_to_realized,
    }

if __name__ == "__main__":
    market = get_sol_market_data()

    # Вставь сюда MVRV из внешнего источника, например Glassnode
    # пример:
    mvrv_input = "0.65081472"

    result = compute_from_mvrv(
        price=market["price_usd"],
        market_cap=market["market_cap_usd"],
        circulating_supply=market["circulating_supply"],
        mvrv=mvrv_input,
    )

    print("Asset:", market["name"], f"({market['symbol'].upper()})")
    print("Price USD:", f"{market['price_usd']:.8f}")
    print("Market Cap USD:", f"{market['market_cap_usd']:.2f}")
    print("Circulating Supply:", f"{market['circulating_supply']:.8f}")
    print("MVRV:", f"{result['mvrv']:.8f}")
    print("Realized Cap USD:", f"{result['realized_cap_usd']:.2f}")
    print("Realized Price USD:", f"{result['realized_price_usd']:.8f}")
    print("Discount to Realized:", f"{result['discount_to_realized']:.2%}")

Asset: Solana (SOL)
Price USD: 84.42000000
Market Cap USD: 48426441006.00
Circulating Supply: 574204518.93391450
MVRV: 0.65081472
Realized Cap USD: 74408951607.61
Realized Price USD: 129.71433713
Discount to Realized: -34.92%


In [7]:
from dataclasses import dataclass, asdict


@dataclass
class MarketInputs:
    price: float
    realized_price: float
    stable_score: float        # 0..100
    dominance_score: float     # 0..100
    treasury_3m_yield: float   # например 3.678
    fed_rate: float            # например 3.75


@dataclass
class MarketResult:
    mvrv: float
    discount_to_realized_pct: float
    mvrv_score: float
    macro_score: float
    stable_score: float
    dominance_score: float
    final_score: float
    regime: str


def clamp(value: float, low: float = 0.0, high: float = 100.0) -> float:
    return max(low, min(high, value))


def compute_mvrv(price: float, realized_price: float) -> float:
    if realized_price <= 0:
        raise ValueError("realized_price must be > 0")
    return price / realized_price


def compute_discount_to_realized(price: float, realized_price: float) -> float:
    return (price / realized_price - 1.0) * 100.0


def score_mvrv(mvrv: float) -> float:
    """
    Чем ниже MVRV, тем выше score.
    Примерная шкала:
    <= 0.60  -> 95
    1.00     -> 80
    1.50     -> 50
    2.00     -> 20
    >= 2.50  -> 5
    """
    if mvrv <= 0.60:
        return 95.0
    elif mvrv <= 1.00:
        # 0.60..1.00 => 95..80
        return 95.0 - (mvrv - 0.60) / 0.40 * 15.0
    elif mvrv <= 1.50:
        # 1.00..1.50 => 80..50
        return 80.0 - (mvrv - 1.00) / 0.50 * 30.0
    elif mvrv <= 2.00:
        # 1.50..2.00 => 50..20
        return 50.0 - (mvrv - 1.50) / 0.50 * 30.0
    elif mvrv <= 2.50:
        # 2.00..2.50 => 20..5
        return 20.0 - (mvrv - 2.00) / 0.50 * 15.0
    else:
        return 5.0


def score_macro(treasury_3m_yield: float, fed_rate: float) -> float:
    """
    Логика:
    - чем ниже ставка и 3M yield, тем лучше для риска
    - чем меньше спред между ставкой и 3M, тем менее "строгий" кэш-режим
    """
    # Базовый score от уровня доходности 3M
    # 2% -> отлично, 5.5% -> плохо
    yield_score = 100.0 - ((treasury_3m_yield - 2.0) / (5.5 - 2.0)) * 100.0
    yield_score = clamp(yield_score)

    # Базовый score от ставки ФРС
    # 2% -> отлично, 5.5% -> плохо
    fed_score = 100.0 - ((fed_rate - 2.0) / (5.5 - 2.0)) * 100.0
    fed_score = clamp(fed_score)

    # Спред: если 3M почти равен ставке, это "нормальная" жёсткая денежная среда
    # если заметно ниже ставки, можно трактовать как небольшое смягчение на коротком конце
    spread = fed_rate - treasury_3m_yield

    # Небольшой бонус за то, что 3M торгуется ниже ставки
    # при спреде 0.0 бонус 0
    # при спреде 0.25 бонус ~5
    spread_bonus = clamp((spread / 0.25) * 5.0, 0.0, 8.0)

    # Смешиваем
    macro_score = 0.55 * yield_score + 0.45 * fed_score + spread_bonus
    return clamp(macro_score)


def classify_regime(score: float) -> str:
    if score >= 80:
        return "Aggressive Buy / Strong Risk-On"
    elif score >= 60:
        return "Accumulate / Mild Risk-On"
    elif score >= 40:
        return "Neutral / Wait"
    else:
        return "Defensive / Risk-Off"


def compute_market_score(inputs: MarketInputs) -> MarketResult:
    mvrv = compute_mvrv(inputs.price, inputs.realized_price)
    discount_pct = compute_discount_to_realized(inputs.price, inputs.realized_price)

    mvrv_score = score_mvrv(mvrv)
    macro_score = score_macro(inputs.treasury_3m_yield, inputs.fed_rate)

    final_score = (
        0.35 * mvrv_score
        + 0.25 * clamp(inputs.stable_score)
        + 0.20 * clamp(inputs.dominance_score)
        + 0.20 * macro_score
    )

    regime = classify_regime(final_score)

    return MarketResult(
        mvrv=round(mvrv, 6),
        discount_to_realized_pct=round(discount_pct, 2),
        mvrv_score=round(mvrv_score, 2),
        macro_score=round(macro_score, 2),
        stable_score=round(clamp(inputs.stable_score), 2),
        dominance_score=round(clamp(inputs.dominance_score), 2),
        final_score=round(final_score, 2),
        regime=regime,
    )


if __name__ == "__main__":
    # Пример на базе твоих чисел
    inputs = MarketInputs(
        price=84.05,
        realized_price=129.14581895,
        stable_score=80,          # много ликвидности в стейблах
        dominance_score=55,       # смешанная картина потоков
        treasury_3m_yield=3.678,  # ты дал это значение
        fed_rate=3.75             # ты дал это значение
    )

    result = compute_market_score(inputs)

    print("=== INPUTS ===")
    for k, v in asdict(inputs).items():
        print(f"{k}: {v}")

    print("\n=== RESULT ===")
    for k, v in asdict(result).items():
        print(f"{k}: {v}")

=== INPUTS ===
price: 84.05
realized_price: 129.14581895
stable_score: 80
dominance_score: 55
treasury_3m_yield: 3.678
fed_rate: 3.75

=== RESULT ===
mvrv: 0.650815
discount_to_realized_pct: -34.92
mvrv_score: 93.09
macro_score: 52.57
stable_score: 80
dominance_score: 55
final_score: 74.1
regime: Accumulate / Mild Risk-On


In [8]:
import requests
from datetime import datetime, timezone
from pprint import pprint


DEFI_LLAMA_STABLES_URL = "https://stablecoins.llama.fi/stablecoincharts/all"
COINGECKO_GLOBAL_URL = "https://api.coingecko.com/api/v3/global"


def fetch_json(url: str):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()


def get_total_crypto_market_cap_usd() -> float:
    data = fetch_json(COINGECKO_GLOBAL_URL)

    if "data" not in data or "total_market_cap" not in data["data"]:
        raise RuntimeError(f"Unexpected CoinGecko response: {data}")

    return float(data["data"]["total_market_cap"]["usd"])


def get_stablecoin_series():
    data = fetch_json(DEFI_LLAMA_STABLES_URL)

    if not isinstance(data, list) or not data:
        raise RuntimeError(f"Unexpected DeFiLlama response: {data}")

    return data


def pick_latest_and_30d_ago(series):
    if len(series) < 31:
        raise RuntimeError("Not enough history to compute 30d change")

    latest = series[-1]
    prev_30d = series[-31]
    return latest, prev_30d


def extract_numeric_value(value):
    """Преобразует число или вложенный dict в float."""
    if isinstance(value, (int, float)):
        return float(value)

    if isinstance(value, str):
        return float(value)

    if isinstance(value, dict):
        # самые вероятные ключи
        for key in [
            "peggedUSD",
            "usd",
            "value",
            "total",
            "circulating",
            "totalCirculatingUSD",
        ]:
            if key in value and isinstance(value[key], (int, float, str)):
                return float(value[key])

        # если ключи неизвестны — пробуем первое числовое значение
        for v in value.values():
            if isinstance(v, (int, float, str)):
                try:
                    return float(v)
                except ValueError:
                    pass

    raise RuntimeError(f"Cannot extract numeric value from: {value}")


def extract_total_stablecoin_market_cap(entry: dict) -> float:
    """
    Пытаемся достать total stablecoin cap из записи DeFiLlama,
    даже если формат слегка отличается.
    """
    possible_keys = [
        "totalCirculatingUSD",
        "totalCirculating",
        "circulatingUSD",
        "circulating",
        "total",
        "value",
    ]

    for key in possible_keys:
        if key in entry:
            return extract_numeric_value(entry[key])

    raise RuntimeError(f"Unexpected stablecoin entry format: {entry}")


def format_ts(ts):
    if ts is None:
        return "n/a"

    try:
        ts = float(ts)
        if ts > 10_000_000_000:
            ts /= 1000.0
        return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat()
    except Exception:
        return str(ts)


def main():
    total_crypto_market_cap = get_total_crypto_market_cap_usd()
    stable_series = get_stablecoin_series()

    latest, prev_30d = pick_latest_and_30d_ago(stable_series)

    print("=== Latest entry sample ===")
    pprint(latest)

    total_stablecoin_market_cap = extract_total_stablecoin_market_cap(latest)
    total_stablecoin_market_cap_30d_ago = extract_total_stablecoin_market_cap(prev_30d)

    stablecoin_30d_change = (
        (total_stablecoin_market_cap - total_stablecoin_market_cap_30d_ago)
        / total_stablecoin_market_cap_30d_ago
        * 100.0
    )

    stablecoin_share = total_stablecoin_market_cap / total_crypto_market_cap * 100.0

    result = {
        "stablecoin_market_cap_usd": round(total_stablecoin_market_cap, 2),
        "stablecoin_market_cap_30d_ago_usd": round(total_stablecoin_market_cap_30d_ago, 2),
        "stablecoin_30d_change_pct": round(stablecoin_30d_change, 2),
        "total_crypto_market_cap_usd": round(total_crypto_market_cap, 2),
        "stablecoin_share_pct": round(stablecoin_share, 2),
        "stablecoins_latest_timestamp": format_ts(latest.get("date")),
        "stablecoins_30d_ago_timestamp": format_ts(prev_30d.get("date")),
    }

    print("\n=== Stablecoin Liquidity Metrics ===")
    for k, v in result.items():
        print(f"{k}: {v}")


if __name__ == "__main__":
    main()

=== Latest entry sample ===
{'date': '1775692800',
 'totalBridgedToUSD': {'peggedARS': 0,
                       'peggedAUD': 0,
                       'peggedCAD': 0,
                       'peggedCHF': 0,
                       'peggedCNY': 0,
                       'peggedCOP': 0,
                       'peggedEUR': 0,
                       'peggedGBP': 0,
                       'peggedGHS': 0,
                       'peggedJPY': 0,
                       'peggedKES': 0,
                       'peggedMXN': 0,
                       'peggedNGN': 0,
                       'peggedPHP': 0,
                       'peggedREAL': 0,
                       'peggedRUB': 0,
                       'peggedSGD': 0,
                       'peggedTRY': 0,
                       'peggedUAH': 0,
                       'peggedUSD': 0,
                       'peggedVAR': 0,
                       'peggedXOF': 0,
                       'peggedZAR': 0},
 'totalCirculating': {'peggedARS': 10898765.87,
  

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class MarketInputs:
    price: float
    realized_price: float
    btc_dominance: float
    sol_dominance: float
    stablecoin_share: float
    stablecoin_30d_change: float
    treasury_3m_yield: float
    fed_rate: float

def clamp(x, lo=0.0, hi=100.0):
    return max(lo, min(hi, x))

def compute_mvrv(price: float, realized_price: float) -> float:
    if realized_price <= 0:
        raise ValueError("realized_price must be > 0")
    return price / realized_price

def score_mvrv(mvrv: float) -> float:
    if mvrv <= 0.60:
        return 95.0
    elif mvrv <= 1.00:
        return 95.0 - (mvrv - 0.60) / 0.40 * 15.0
    elif mvrv <= 1.50:
        return 80.0 - (mvrv - 1.00) / 0.50 * 30.0
    elif mvrv <= 2.00:
        return 50.0 - (mvrv - 1.50) / 0.50 * 30.0
    elif mvrv <= 2.50:
        return 20.0 - (mvrv - 2.00) / 0.50 * 15.0
    return 5.0

def score_macro(treasury_3m_yield: float, fed_rate: float) -> float:
    yield_score = 100.0 - ((treasury_3m_yield - 2.0) / (5.5 - 2.0)) * 100.0
    fed_score = 100.0 - ((fed_rate - 2.0) / (5.5 - 2.0)) * 100.0
    spread = fed_rate - treasury_3m_yield
    spread_bonus = clamp((spread / 0.25) * 5.0, 0.0, 8.0)
    return clamp(0.55 * clamp(yield_score) + 0.45 * clamp(fed_score) + spread_bonus)

def calc_stable_score(stablecoin_share: float, stablecoin_30d_change: float) -> float:
    base = 50 + (stablecoin_share - 10.0) * 8
    trend = stablecoin_30d_change * 10
    return clamp(base + trend)

def calc_dominance_score(sol_dominance: float, btc_dominance: float, stablecoin_share: float) -> float:
    base = 50
    sol_part = (sol_dominance - 1.5) * 20
    btc_part = -(btc_dominance - 55.0) * 3
    stable_part = -(stablecoin_share - 10.0) * 2
    return clamp(base + sol_part + btc_part + stable_part)

def classify(score: float) -> str:
    if score >= 80:
        return "Aggressive Buy / Strong Risk-On"
    elif score >= 60:
        return "Accumulate / Mild Risk-On"
    elif score >= 40:
        return "Neutral / Wait"
    return "Defensive / Risk-Off"

def compute_market_score(inputs: MarketInputs):
    mvrv = compute_mvrv(inputs.price, inputs.realized_price)
    mvrv_score = score_mvrv(mvrv)
    stable_score = calc_stable_score(inputs.stablecoin_share, inputs.stablecoin_30d_change)
    dominance_score = calc_dominance_score(inputs.sol_dominance, inputs.btc_dominance, inputs.stablecoin_share)
    macro_score = score_macro(inputs.treasury_3m_yield, inputs.fed_rate)

    final_score = (
        0.35 * mvrv_score
        + 0.25 * stable_score
        + 0.20 * dominance_score
        + 0.20 * macro_score
    )

    return {
        "mvrv": round(mvrv, 6),
        "mvrv_score": round(mvrv_score, 2),
        "stable_score": round(stable_score, 2),
        "dominance_score": round(dominance_score, 2),
        "macro_score": round(macro_score, 2),
        "final_score": round(final_score, 2),
        "regime": classify(final_score),
    }

inputs = MarketInputs(
    price=84.05,
    realized_price=129.14581895,
    btc_dominance=59.48,
    sol_dominance=1.993,
    stablecoin_share=12.47,
    stablecoin_30d_change=1.14,
    treasury_3m_yield=3.678,
    fed_rate=3.75,
)

print(compute_market_score(inputs))

{'mvrv': 0.650815, 'mvrv_score': 93.09, 'stable_score': 69.88, 'dominance_score': 47.48, 'macro_score': 52.57, 'final_score': 70.06, 'regime': 'Accumulate / Mild Risk-On'}


In [6]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Model v2: Macro + Liquidity + Flows + Valuation + Momentum

Sources:
- CoinGecko: global market data, coin market data
- DeFiLlama: stablecoin total history
- FRED: DTB3, DFEDTARU
- Binance Futures: funding history, open interest, open interest history

Usage:
    python model_v2.py --realized-price 128.09150996
or:
    python model_v2.py --mvrv 0.65749869

Notes:
- TOTAL3 is approximated as Total Crypto Market Cap - BTC Market Cap - ETH Market Cap
- funding/open interest use SOLUSDT perpetual on Binance as market proxy
"""

from __future__ import annotations

import argparse
import csv
import io
import math
import statistics
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone
from typing import Any
import time

import requests
from requests import exceptions as requests_exceptions


# -----------------------------
# Endpoints
# -----------------------------
COINGECKO_GLOBAL_URL = "https://api.coingecko.com/api/v3/global"
COINGECKO_MARKETS_URL = "https://api.coingecko.com/api/v3/coins/markets"

DEFILLAMA_STABLES_URL = "https://stablecoins.llama.fi/stablecoincharts/all"

FRED_CSV_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv"

BINANCE_FUNDING_URL = "https://fapi.binance.com/fapi/v1/fundingRate"
BINANCE_OPEN_INTEREST_URL = "https://fapi.binance.com/fapi/v1/openInterest"
BINANCE_OPEN_INTEREST_HIST_URL = "https://fapi.binance.com/futures/data/openInterestHist"


# -----------------------------
# Utilities
# -----------------------------
_HTTP_CACHE: dict[tuple[str, tuple[tuple[str, str], ...]], Any] = {}


def _cache_key(url: str, params: dict[str, Any] | None = None) -> tuple[str, tuple[tuple[str, str], ...]]:
    normalized = tuple(sorted((params or {}).items()))
    return url, tuple((str(k), str(v)) for k, v in normalized)


def _request_with_retry(url: str, params: dict[str, Any] | None = None) -> requests.Response:
    last_error = None
    for attempt in range(4):
        resp = requests.get(url, params=params, timeout=30)

        if resp.status_code != 429:
            try:
                resp.raise_for_status()
                return resp
            except requests_exceptions.RequestException as exc:
                last_error = exc
                break

        retry_after = resp.headers.get("Retry-After")
        if retry_after is not None:
            try:
                delay = max(float(retry_after), 1.0)
            except ValueError:
                delay = float(2 ** attempt)
        else:
            delay = float(2 ** attempt)

        last_error = requests_exceptions.HTTPError(
            f"429 Too Many Requests for url: {resp.url}",
            response=resp,
        )

        if attempt == 3:
            break

        time.sleep(delay)

    raise RuntimeError(
        "Request failed after retries. CoinGecko may be rate-limiting this notebook; "
        "please wait a bit and rerun the cell. "
        f"Original error: {last_error}"
    ) from last_error


def fetch_json(url: str, params: dict[str, Any] | None = None) -> Any:
    key = _cache_key(url, params)
    if key in _HTTP_CACHE:
        return _HTTP_CACHE[key]

    data = _request_with_retry(url, params).json()
    _HTTP_CACHE[key] = data
    return data


def fetch_text(url: str, params: dict[str, Any] | None = None) -> str:
    key = _cache_key(url, params)
    if key in _HTTP_CACHE:
        return _HTTP_CACHE[key]

    text = _request_with_retry(url, params).text
    _HTTP_CACHE[key] = text
    return text


def clamp(x: float, lo: float = 0.0, hi: float = 100.0) -> float:
    return max(lo, min(hi, x))


def pct_change(current: float, previous: float) -> float:
    if previous == 0:
        raise ZeroDivisionError("previous value is zero")
    return (current - previous) / previous * 100.0


def mean_or_none(values: list[float]) -> float | None:
    vals = [v for v in values if v is not None]
    if not vals:
        return None
    return sum(vals) / len(vals)


# -----------------------------
# FRED
# -----------------------------
def fred_latest_value(series_id: str) -> float:
    """
    Pull latest non-empty value from FRED CSV graph endpoint.
    """
    csv_text = fetch_text(FRED_CSV_URL, {"id": series_id})
    reader = csv.DictReader(io.StringIO(csv_text))

    latest = None
    for row in reader:
        value = row.get(series_id)
        if value and value != ".":
            latest = float(value)

    if latest is None:
        raise RuntimeError(f"No valid FRED value found for {series_id}")

    return latest


# -----------------------------
# CoinGecko
# -----------------------------
def get_global_market_data() -> dict[str, float]:
    data = fetch_json(COINGECKO_GLOBAL_URL)["data"]
    return {
        "total_market_cap_usd": float(data["total_market_cap"]["usd"]),
        "btc_dominance": float(data["market_cap_percentage"]["btc"]),
        "eth_dominance": float(data["market_cap_percentage"]["eth"]),
    }


def get_coin_markets(ids: list[str]) -> dict[str, dict[str, Any]]:
    data = fetch_json(
        COINGECKO_MARKETS_URL,
        {
            "vs_currency": "usd",
            "ids": ",".join(ids),
        },
    )
    return {row["id"]: row for row in data}


# -----------------------------
# DeFiLlama stablecoin history
# -----------------------------
def get_stablecoin_history() -> list[dict[str, Any]]:
    data = fetch_json(DEFILLAMA_STABLES_URL)
    if not isinstance(data, list) or not data:
        raise RuntimeError("Unexpected DeFiLlama response")
    return data


def extract_stable_total(entry: dict[str, Any]) -> float:
    """
    Handles multiple possible payload shapes.
    """
    # Most common shape
    if "totalCirculatingUSD" in entry:
        val = entry["totalCirculatingUSD"]
        if isinstance(val, (int, float, str)):
            return float(val)
        if isinstance(val, dict):
            for key in ("peggedUSD", "usd", "value", "total"):
                if key in val:
                    return float(val[key])

    # Fallback: first numeric-ish field
    for v in entry.values():
        if isinstance(v, (int, float, str)):
            try:
                return float(v)
            except ValueError:
                pass
        if isinstance(v, dict):
            for subv in v.values():
                if isinstance(subv, (int, float, str)):
                    try:
                        return float(subv)
                    except ValueError:
                        pass

    raise RuntimeError(f"Cannot parse stablecoin total from entry: {entry}")


def get_stablecoin_metrics() -> dict[str, float]:
    series = get_stablecoin_history()
    if len(series) < 31:
        raise RuntimeError("Not enough DeFiLlama history to compute 30d change")

    latest = extract_stable_total(series[-1])
    prev_30d = extract_stable_total(series[-31])

    return {
        "stablecoin_market_cap_usd": latest,
        "stablecoin_30d_change": pct_change(latest, prev_30d),
    }


# -----------------------------
# Binance Futures
# -----------------------------
def get_latest_funding_rate(symbol: str = "SOLUSDT") -> float:
    data = fetch_json(BINANCE_FUNDING_URL, {"symbol": symbol, "limit": 5})
    if not data:
        raise RuntimeError("Empty funding history from Binance")
    return float(data[-1]["fundingRate"]) * 100.0  # convert to percent


def get_open_interest(symbol: str = "SOLUSDT") -> float:
    data = fetch_json(BINANCE_OPEN_INTEREST_URL, {"symbol": symbol})
    return float(data["openInterest"])


def get_open_interest_change_7d(symbol: str = "SOLUSDT") -> float:
    data = fetch_json(
        BINANCE_OPEN_INTEREST_HIST_URL,
        {
            "symbol": symbol,
            "period": "1d",
            "limit": 8,
        },
    )
    if len(data) < 8:
        raise RuntimeError("Not enough open interest history from Binance")

    latest = float(data[-1]["sumOpenInterest"])
    prev_7d = float(data[0]["sumOpenInterest"])
    return pct_change(latest, prev_7d)


# -----------------------------
# Derived market inputs
# -----------------------------
@dataclass
class MarketInputsV2:
    price: float
    realized_price: float
    mvrv: float

    fed_rate: float
    treasury_3m_yield: float

    stablecoin_market_cap_usd: float
    stablecoin_share: float
    stablecoin_30d_change: float

    total_market_cap_usd: float
    total3_market_cap_usd: float
    total3_30d_change: float

    btc_dominance: float
    eth_dominance: float
    sol_dominance: float
    btc_dominance_30d_change: float
    sol_dominance_30d_change: float

    funding_rate: float              # percent per funding interval
    open_interest: float
    open_interest_change_7d: float


@dataclass
class MarketResultV2:
    discount_to_realized_pct: float

    mvrv_score: float
    stable_score: float
    macro_score: float
    dominance_score: float
    dominance_trend_score: float
    total3_score: float
    momentum_score: float

    final_score: float
    regime: str
    allocation_total_risk_pct: int
    allocation_stablecoins_pct: int
    allocation_majors_pct: int
    allocation_alts_pct: int


def compute_realized_price(price: float, mvrv: float) -> float:
    if mvrv <= 0:
        raise ValueError("MVRV must be > 0")
    return price / mvrv


def get_current_market_snapshot() -> dict[str, float]:
    global_data = get_global_market_data()
    coins = get_coin_markets(["solana", "bitcoin", "ethereum"])

    total_market_cap = global_data["total_market_cap_usd"]
    btc_mc = float(coins["bitcoin"]["market_cap"])
    eth_mc = float(coins["ethereum"]["market_cap"])
    sol_mc = float(coins["solana"]["market_cap"])

    price = float(coins["solana"]["current_price"])

    total3_mc = total_market_cap - btc_mc - eth_mc
    sol_dom = sol_mc / total_market_cap * 100.0

    return {
        **global_data,
        "price": price,
        "btc_market_cap_usd": btc_mc,
        "eth_market_cap_usd": eth_mc,
        "sol_market_cap_usd": sol_mc,
        "sol_dominance": sol_dom,
        "total3_market_cap_usd": total3_mc,
    }


def get_historical_coin_market_cap(coin_id: str, days_back: int) -> float:
    """
    CoinGecko /coins/{id}/history is date-based; easier to use /coins/markets
    is current only, so for 30d historical we use /coins/{id}/history with date.
    """
    target_date = datetime.now(timezone.utc) - timedelta(days=days_back)
    date_str = target_date.strftime("%d-%m-%Y")

    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/history"
    data = fetch_json(url, {"date": date_str, "localization": "false"})

    mc = data.get("market_data", {}).get("market_cap", {}).get("usd")
    if mc is None:
        raise RuntimeError(f"Could not get historical market cap for {coin_id} at {date_str}")
    return float(mc)


def get_historical_global_total_market_cap(days_back: int) -> float:
    """
    CoinGecko free global endpoint is current only, so for a simple 30d proxy
    we reconstruct 30d total market cap from current snapshot proportions using
    the major-coin history plus current dominance as fallback is weak.
    Better: user can later swap this with a paid historical global endpoint.

    Here we use CoinGecko historical market caps for BTC and ETH plus
    /coins/history for SOL, and approximate past total from BTC dominance today
    is NOT good enough, so instead we rely on an optional manual method below.

    To keep the script usable on free endpoints, we estimate past TOTAL3 via
    current market snapshot and historical BTC/ETH market caps using current total
    market cap minus current BTC/ETH, then compare to historical BTC/ETH move.
    This is a proxy, not a canonical metric.
    """
    raise NotImplementedError


def get_total3_30d_change(
    current_total_market_cap: float,
    current_btc_mc: float,
    current_eth_mc: float,
    current_sol_mc: float,
) -> float:
    """
    Approximate TOTAL3 30d change using historical BTC/ETH and historical SOL.
    Since free historical global market cap is limited, we approximate:
    historical_total3 ~= historical market cap of top alt sector proxy:
    current TOTAL3 scaled by historical change in TOTAL market represented by
    BTC+ETH+SOL basket.
    """
    # Use a proxy basket: BTC, ETH, SOL 30d change vs current
    btc_30 = get_historical_coin_market_cap("bitcoin", 30)
    eth_30 = get_historical_coin_market_cap("ethereum", 30)
    sol_30 = get_historical_coin_market_cap("solana", 30)

    basket_now = current_btc_mc + current_eth_mc + current_sol_mc
    basket_now = float(basket_now)
    basket_30 = btc_30 + eth_30 + sol_30

    if basket_30 <= 0:
        raise RuntimeError("Invalid historical basket for TOTAL3 proxy")

    basket_change = pct_change(basket_now, basket_30)

    current_total3 = current_total_market_cap - current_btc_mc - current_eth_mc
    # Proxy historical total3
    historical_total3_proxy = current_total3 / (1 + basket_change / 100.0)
    return pct_change(current_total3, historical_total3_proxy)


def get_dominance_trends(
    current_total_market_cap: float,
    current_btc_mc: float,
    current_eth_mc: float,
    current_sol_mc: float,
) -> dict[str, float]:
    btc_mc_now = float(current_btc_mc)
    sol_mc_now = float(current_sol_mc)

    btc_mc_30 = get_historical_coin_market_cap("bitcoin", 30)
    sol_mc_30 = get_historical_coin_market_cap("solana", 30)

    # Approximate total market 30d ago using basket ratio
    eth_mc_now = float(current_eth_mc)
    eth_mc_30 = get_historical_coin_market_cap("ethereum", 30)

    basket_now = btc_mc_now + eth_mc_now + sol_mc_now
    basket_30 = btc_mc_30 + eth_mc_30 + sol_mc_30

    approx_total_30 = current_total_market_cap / (1 + pct_change(basket_now, basket_30) / 100.0)

    btc_dom_now = btc_mc_now / current_total_market_cap * 100.0
    sol_dom_now = sol_mc_now / current_total_market_cap * 100.0

    btc_dom_30 = btc_mc_30 / approx_total_30 * 100.0
    sol_dom_30 = sol_mc_30 / approx_total_30 * 100.0

    return {
        "btc_dominance_30d_change": btc_dom_now - btc_dom_30,
        "sol_dominance_30d_change": sol_dom_now - sol_dom_30,
    }


# -----------------------------
# Scoring
# -----------------------------
def score_mvrv(mvrv: float) -> float:
    if mvrv <= 0.60:
        return 95.0
    elif mvrv <= 1.00:
        return 95.0 - (mvrv - 0.60) / 0.40 * 15.0
    elif mvrv <= 1.50:
        return 80.0 - (mvrv - 1.00) / 0.50 * 30.0
    elif mvrv <= 2.00:
        return 50.0 - (mvrv - 1.50) / 0.50 * 30.0
    elif mvrv <= 2.50:
        return 20.0 - (mvrv - 2.00) / 0.50 * 15.0
    return 5.0


def score_macro(treasury_3m_yield: float, fed_rate: float) -> float:
    yield_score = 100.0 - ((treasury_3m_yield - 2.0) / (5.5 - 2.0)) * 100.0
    fed_score = 100.0 - ((fed_rate - 2.0) / (5.5 - 2.0)) * 100.0
    spread = fed_rate - treasury_3m_yield
    spread_bonus = clamp((spread / 0.25) * 5.0, 0.0, 8.0)
    return clamp(0.55 * clamp(yield_score) + 0.45 * clamp(fed_score) + spread_bonus)


def score_stable(stablecoin_share: float, stablecoin_30d_change: float) -> float:
    base = 50 + (stablecoin_share - 10.0) * 8.0
    trend = stablecoin_30d_change * 10.0
    return clamp(base + trend)


def score_dominance_level(sol_dominance: float, btc_dominance: float, stablecoin_share: float) -> float:
    base = 50.0
    sol_part = (sol_dominance - 1.5) * 20.0
    btc_part = -(btc_dominance - 55.0) * 3.0
    stable_part = -(stablecoin_share - 10.0) * 2.0
    return clamp(base + sol_part + btc_part + stable_part)


def score_dominance_trend(btc_d_change_30d: float, sol_d_change_30d: float) -> float:
    score = 50.0
    score += (-btc_d_change_30d) * 6.0   # falling BTC.D is risk-on
    score += sol_d_change_30d * 10.0     # rising SOL.D favors the asset
    return clamp(score)


def score_total3(total3_30d_change: float) -> float:
    return clamp(50.0 + total3_30d_change * 2.0)


def score_momentum(funding_rate_pct: float, oi_change_7d: float) -> float:
    score = 50.0

    # funding is percent; e.g. 0.01 means +0.01%
    if funding_rate_pct < 0:
        score += 10.0
    elif funding_rate_pct > 0.10:
        score -= 20.0
    elif funding_rate_pct > 0.03:
        score -= 8.0
    else:
        score += 5.0

    score += oi_change_7d * 1.2
    return clamp(score)


def apply_funding_guardrails(final_score: float, funding_rate_pct: float) -> tuple[float, str | None]:
    """
    Funding is stored in percent units, e.g. 0.10 means +0.10% per interval.
    This applies an additional tactical haircut when perp positioning looks overheated.
    """
    adjusted = final_score
    regime_override = None

    if funding_rate_pct > 0.10:
        funding_penalty = min(0.40, (funding_rate_pct - 0.10) * 0.50)
        adjusted *= 1.0 - funding_penalty

    if funding_rate_pct > 0.15 and adjusted > 65:
        adjusted *= 0.90
        regime_override = "Cautious Accumulation (High Funding)"

    return adjusted, regime_override


def apply_risk_allocation_guardrails(
    total_risk: int,
    stablecoins: int,
    majors_pct: int,
    alts_pct: int,
    funding_rate_pct: float,
    final_score: float,
) -> tuple[int, int, int, int]:
    adjusted_risk = float(total_risk)

    if funding_rate_pct > 0.10:
        funding_penalty = min(0.40, (funding_rate_pct - 0.10) * 0.50)
        adjusted_risk *= 1.0 - funding_penalty * 0.70

    if funding_rate_pct > 0.15 and final_score > 65:
        adjusted_risk *= 0.70

    adjusted_risk = max(0.0, min(100.0, adjusted_risk))
    stablecoins = int(round(100.0 - adjusted_risk))
    total_risk = int(round(adjusted_risk))

    if total_risk == 0:
        return 0, 100, 0, 0

    majors_share = majors_pct / max(1, majors_pct + alts_pct)
    majors = int(round(total_risk * majors_share))
    alts = total_risk - majors
    return total_risk, stablecoins, majors, alts


def classify_regime(score: float) -> str:
    if score >= 80:
        return "Strong Risk-On"
    elif score >= 65:
        return "Accumulate / Mild Risk-On"
    elif score >= 50:
        return "Neutral"
    elif score >= 35:
        return "Defensive"
    return "Risk-Off"


def allocation_from_score(score: float) -> tuple[int, int, int, int]:
    """
    Returns:
    total_risk, stablecoins, majors, alts
    """
    if score >= 80:
        total_risk, stable = 90, 10
    elif score >= 65:
        total_risk, stable = 70, 30
    elif score >= 50:
        total_risk, stable = 50, 50
    elif score >= 35:
        total_risk, stable = 30, 70
    else:
        total_risk, stable = 15, 85

    if score > 75:
        majors, alts = 40, 60
    elif score > 60:
        majors, alts = 60, 40
    else:
        majors, alts = 80, 20

    majors_pct = round(total_risk * majors / 100.0)
    alts_pct = round(total_risk * alts / 100.0)
    return total_risk, stable, majors_pct, alts_pct


# -----------------------------
# Main compute
# -----------------------------
def build_inputs(realized_price: float | None, mvrv: float | None) -> MarketInputsV2:
    snapshot = get_current_market_snapshot()
    stable = get_stablecoin_metrics()

    fed_rate = fred_latest_value("DFEDTARU")
    treasury_3m_yield = fred_latest_value("DTB3")

    funding_rate = get_latest_funding_rate("SOLUSDT")
    open_interest = get_open_interest("SOLUSDT")
    oi_change_7d = get_open_interest_change_7d("SOLUSDT")

    if realized_price is None:
        if mvrv is None:
            raise ValueError("Provide either --realized-price or --mvrv")
        realized_price = compute_realized_price(snapshot["price"], mvrv)
    else:
        mvrv = snapshot["price"] / realized_price

    total3_30d_change = get_total3_30d_change(
        current_total_market_cap=snapshot["total_market_cap_usd"],
        current_btc_mc=snapshot["btc_market_cap_usd"],
        current_eth_mc=snapshot["eth_market_cap_usd"],
        current_sol_mc=snapshot["sol_market_cap_usd"],
    )

    dom_trends = get_dominance_trends(
        current_total_market_cap=snapshot["total_market_cap_usd"],
        current_btc_mc=snapshot["btc_market_cap_usd"],
        current_eth_mc=snapshot["eth_market_cap_usd"],
        current_sol_mc=snapshot["sol_market_cap_usd"],
    )

    stable_share = stable["stablecoin_market_cap_usd"] / snapshot["total_market_cap_usd"] * 100.0

    return MarketInputsV2(
        price=snapshot["price"],
        realized_price=realized_price,
        mvrv=mvrv,

        fed_rate=fed_rate,
        treasury_3m_yield=treasury_3m_yield,

        stablecoin_market_cap_usd=stable["stablecoin_market_cap_usd"],
        stablecoin_share=stable_share,
        stablecoin_30d_change=stable["stablecoin_30d_change"],

        total_market_cap_usd=snapshot["total_market_cap_usd"],
        total3_market_cap_usd=snapshot["total3_market_cap_usd"],
        total3_30d_change=total3_30d_change,

        btc_dominance=snapshot["btc_dominance"],
        eth_dominance=snapshot["eth_dominance"],
        sol_dominance=snapshot["sol_dominance"],
        btc_dominance_30d_change=dom_trends["btc_dominance_30d_change"],
        sol_dominance_30d_change=dom_trends["sol_dominance_30d_change"],

        funding_rate=funding_rate,
        open_interest=open_interest,
        open_interest_change_7d=oi_change_7d,
    )


def compute_result(inputs: MarketInputsV2) -> MarketResultV2:
    discount = (inputs.price / inputs.realized_price - 1.0) * 100.0

    mvrv_score = score_mvrv(inputs.mvrv)
    stable_score = score_stable(inputs.stablecoin_share, inputs.stablecoin_30d_change)
    macro_score = score_macro(inputs.treasury_3m_yield, inputs.fed_rate)
    dominance_score = score_dominance_level(
        inputs.sol_dominance,
        inputs.btc_dominance,
        inputs.stablecoin_share,
    )
    dominance_trend_score = score_dominance_trend(
        inputs.btc_dominance_30d_change,
        inputs.sol_dominance_30d_change,
    )
    total3_score = score_total3(inputs.total3_30d_change)
    momentum_score = score_momentum(inputs.funding_rate, inputs.open_interest_change_7d)

    final_score = (
        0.25 * mvrv_score +
        0.20 * stable_score +
        0.15 * macro_score +
        0.15 * dominance_score +
        0.10 * dominance_trend_score +
        0.10 * total3_score +
        0.05 * momentum_score
    )

    final_score, regime_override = apply_funding_guardrails(final_score, inputs.funding_rate)
    regime = regime_override or classify_regime(final_score)
    total_risk, stablecoins, majors, alts = allocation_from_score(final_score)
    total_risk, stablecoins, majors, alts = apply_risk_allocation_guardrails(
        total_risk,
        stablecoins,
        majors,
        alts,
        inputs.funding_rate,
        final_score,
    )

    return MarketResultV2(
        discount_to_realized_pct=round(discount, 2),

        mvrv_score=round(mvrv_score, 2),
        stable_score=round(stable_score, 2),
        macro_score=round(macro_score, 2),
        dominance_score=round(dominance_score, 2),
        dominance_trend_score=round(dominance_trend_score, 2),
        total3_score=round(total3_score, 2),
        momentum_score=round(momentum_score, 2),

        final_score=round(final_score, 2),
        regime=regime,
        allocation_total_risk_pct=total_risk,
        allocation_stablecoins_pct=stablecoins,
        allocation_majors_pct=majors,
        allocation_alts_pct=alts,
    )


def print_output(inputs: MarketInputsV2, result: MarketResultV2) -> None:
    print("=== MARKET INPUTS V2 ===")
    for k, v in asdict(inputs).items():
        print(f"{k}: {v}")

    print("\n=== RESULT V2 ===")
    for k, v in asdict(result).items():
        print(f"{k}: {v}")

    print("\n=== INTERPRETATION ===")
    print(f"Regime: {result.regime}")
    print(f"Final score: {result.final_score}")
    print(f"Allocation: risk={result.allocation_total_risk_pct}% | "
          f"stablecoins={result.allocation_stablecoins_pct}% | "
          f"majors={result.allocation_majors_pct}% | "
          f"alts={result.allocation_alts_pct}%")

    print("\n=== REBALANCE RULES ===")
    print("- Rebalance if target allocation differs by >10 percentage points")
    print("- Increase risk only if final_score > 70 and dominance_trend_score > 50")
    print("- Reduce risk if final_score < 60 or funding_rate > 0.10%")
    print("- If funding_rate > 0.15% and final_score stays bullish, apply cautious accumulation haircut")
    print("- Aggressive derisk if final_score < 40")


def main():
    parser = argparse.ArgumentParser(description="Crypto Model v2")
    parser.add_argument("--realized-price", type=float, default=None, help="Direct realized price input")
    parser.add_argument("--mvrv", type=float, default=None, help="If realized price unavailable, derive from MVRV")

    inputs = build_inputs(realized_price=None, mvrv=0.65749869)
    result = compute_result(inputs)
    print_output(inputs, result)


if __name__ == "__main__":
    main()

=== MARKET INPUTS V2 ===
price: 83.96
realized_price: 127.69607191156531
mvrv: 0.65749869
fed_rate: 3.75
treasury_3m_yield: 3.61
stablecoin_market_cap_usd: 316129093177.47
stablecoin_share: 12.530489742098839
stablecoin_30d_change: 1.1379081004075946
total_market_cap_usd: 2522878991037.096
total3_market_cap_usd: 815442030393.0962
total3_30d_change: 5.830874156461225
btc_dominance: 57.10396325962683
eth_dominance: 10.5145933230069
sol_dominance: 1.9157135460203825
btc_dominance_30d_change: -0.31983713636824973
sol_dominance_30d_change: -0.11910104420493539
funding_rate: 0.002462
open_interest: 8654497.82
open_interest_change_7d: -4.834303584294054

=== RESULT V2 ===
discount_to_realized_pct: -34.25
mvrv_score: 92.84
stable_score: 81.62
macro_score: 55.0
dominance_score: 46.94
dominance_trend_score: 50.73
total3_score: 61.66
momentum_score: 49.2
final_score: 68.53
regime: Accumulate / Mild Risk-On
allocation_total_risk_pct: 70
allocation_stablecoins_pct: 30
allocation_majors_pct: 42
allo